In [0]:
# ── Cell 1: Install ────────────────────────────────────────────
%pip install great-expectations==0.18.0

In [0]:
# ── Cell 2: Restart ────────────────────────────────────────────
dbutils.library.restartPython()

In [0]:
# ── Cell 3: Imports ────────────────────────────────────────────
import great_expectations as gx
from great_expectations.data_context.types.base import (
    DataContextConfig,
    FilesystemStoreBackendDefaults
)
from datetime import datetime
import pandas as pd

print(f"GE version : {gx.__version__}")
print(f"Spark      : {spark.version}")

In [0]:
# ── Cell 4: Setup persistent GE context ───────────────────────

GE_ROOT    = "/Volumes/ipl_catalog/raw_data/great_expectations"
SUITE_NAME = "deliveries_suite"
TABLE_NAME = "ipl_catalog.silver.deliveries"

# Create volume
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.great_expectations")

# Create Volume folders
for folder in [
    GE_ROOT,
    f"{GE_ROOT}/expectations",
    f"{GE_ROOT}/validations",
    f"{GE_ROOT}/checkpoints",
    f"{GE_ROOT}/data_docs",
]:
    dbutils.fs.mkdirs(folder)

# Create context with file-based storage
context = gx.get_context(
    project_config = DataContextConfig(
        store_backend_defaults = FilesystemStoreBackendDefaults(
            root_directory = GE_ROOT
        )
    )
)

print(f"GE context ready")
print(f"Suite will be saved to: {GE_ROOT}/expectations/")
print(f"Results saved to      : {GE_ROOT}/validations/")
print(f"Data docs saved to    : {GE_ROOT}/data_docs/")

In [0]:
# ── Cell 5: Load Silver table ──────────────────────────────────
df_silver_spark = spark.table(TABLE_NAME)
total_rows      = df_silver_spark.count()
print(f"Total Silver rows: {total_rows:,}")

# Sample for GE pandas validation
SAMPLE_SIZE = 50_000
df_sample   = (
    df_silver_spark
    .sample(fraction=SAMPLE_SIZE / total_rows, seed=42)
    .toPandas()
)
print(f"Sample size      : {len(df_sample):,}")

In [0]:
# ── Cell 6: Setup datasource and validator ─────────────────────
datasource = context.sources.add_or_update_pandas(
    name = "pandas_datasource"
)

data_asset = datasource.add_dataframe_asset(
    name = "silver_deliveries_sample"
)

batch_request = data_asset.build_batch_request(
    dataframe = df_sample
)

suite = context.add_or_update_expectation_suite(
    expectation_suite_name = SUITE_NAME
)

validator = context.get_validator(
    batch_request          = batch_request,
    expectation_suite_name = SUITE_NAME,
)

print("Validator ready ✓")

In [0]:
# ── Cell 7: Add expectations ───────────────────────────────────

IPL_TEAMS = [
    "Mumbai Indians", "Chennai Super Kings",
    "Royal Challengers Bengaluru", "Kolkata Knight Riders",
    "Delhi Capitals", "Rajasthan Royals",
    "Sunrisers Hyderabad", "Punjab Kings",
    "Lucknow Super Giants", "Gujarat Titans",
    "Kochi Tuskers Kerala", "Pune Warriors",
    "Rising Pune Supergiant", "Gujarat Lions",

    "Kings XI Punjab", "Delhi Daredevils", "Royal Challengers Bangalore", "Rising Pune Supergiants", "Deccan Chargers"
]

# Table level
validator.expect_table_row_count_to_be_between(
    min_value=1_000, max_value=500_000
)

# Not null
for col in ["match_id", "innings", "over_number", "ball_number",
            "batting_team", "bowling_team", "striker",
            "bowler", "runs_off_bat", "total_runs", "phase"]:
    validator.expect_column_values_to_not_be_null(column=col)

# Range checks — innings 1-6 to include Super Overs
validator.expect_column_values_to_be_between(
    column="innings", min_value=1, max_value=6
)
validator.expect_column_values_to_be_between(
    column="over_number", min_value=0, max_value=19
)
validator.expect_column_values_to_be_between(
    column="runs_off_bat", min_value=0, max_value=6
)
validator.expect_column_values_to_be_between(
    column="total_runs", min_value=0, max_value=26
)

# Accepted values
validator.expect_column_values_to_be_in_set(
    column="batting_team", value_set=IPL_TEAMS
)
validator.expect_column_values_to_be_in_set(
    column="bowling_team", value_set=IPL_TEAMS
)
validator.expect_column_values_to_be_in_set(
    column="phase",
    value_set=["powerplay", "middle", "death", "super_over"]
)
validator.expect_column_values_to_be_in_set(
    column="_source", value_set=["cricsheet"]
)
validator.expect_column_values_to_be_in_set(
    column="season",
    value_set=[str(y) for y in range(2007, 2027)]
)

# Boolean checks
for col in ["is_wicket", "is_boundary", "is_six", "is_dot_ball"]:
    validator.expect_column_values_to_be_in_set(
        column=col, value_set=[True, False]
    )

# Consistency
validator.expect_column_pair_values_A_to_be_greater_than_B(
    column_A="total_runs",
    column_B="runs_off_bat",
    or_equal=True,
)

# Save suite to Volume
validator.save_expectation_suite(
    discard_failed_expectations=False
)

print(f"Suite saved to {GE_ROOT}/expectations/{SUITE_NAME}.json")
print(f"Total expectations: {len(validator.get_expectation_suite().expectations)}")

In [0]:
# ── Cell 8: Run validation and save results ────────────────────
results     = validator.validate()
statistics  = results["statistics"]

total       = statistics["evaluated_expectations"]
passed      = statistics["successful_expectations"]
failed      = statistics["unsuccessful_expectations"]
success_pct = statistics["success_percent"]

print(f"Validation Results — {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*45}")
print(f"  Total      : {total}")
print(f"  Passed     : {passed}")
print(f"  Failed     : {failed}")
print(f"  Success %  : {success_pct:.1f}%")

if failed > 0:
    print(f"\nFailed expectations:")
    for result in results.results:
        if not result.success:
            col = result.expectation_config.kwargs.get(
                "column", "table-level"
            )
            exp = result.expectation_config.expectation_type
            print(f"  ✗ {col:30} {exp}")
else:
    print("\n✓ All expectations passed")

# Build data docs — creates browsable HTML report
context.build_data_docs()
print(f"\nData docs built at:")
print(f"  {GE_ROOT}/data_docs/local_site/index.html")

In [0]:
# ── Cell 9: Full table Spark checks ───────────────────────────
# These run on 100% of data — not just the sample

from pyspark.sql.functions import col

df = spark.table(TABLE_NAME)

print("Full-table Spark quality checks:")
print("=" * 45)

checks = {
    "Null match_id"      : df.filter(col("match_id").isNull()).count(),
    "Null batting_team"  : df.filter(col("batting_team").isNull()).count(),
    "Null bowler"        : df.filter(col("bowler").isNull()).count(),
    "Invalid overs"      : df.filter(
                               (col("over_number") < 0) |
                               (col("over_number") > 19)
                           ).count(),
    "Negative runs"      : df.filter(col("runs_off_bat") < 0).count(),
    "Invalid innings"    : df.filter(
                               (col("innings") < 1) |
                               (col("innings") > 6)
                           ).count(),
}

all_passed = True
for check_name, count in checks.items():
    status = "✓" if count == 0 else "✗ FIX NEEDED"
    if count > 0:
        all_passed = False
    print(f"  {check_name:25} count={count:,}  {status}")

if all_passed:
    print("\n✓ All full-table checks passed")
    dbutils.notebook.exit("VALIDATION_PASSED")
else:
    print("\n✗ Some checks failed — review before Gold layer")
    dbutils.notebook.exit("VALIDATION_FAILED")